In [1]:
import cv2
from ultralytics import YOLO
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog
import time


In [2]:
max_frames = 5000
last_feed=[]
ra = False

counter = 0

In [3]:
v = cv2.VideoCapture('C:/Users/HDN/Desktop/Video_survelliance/restricted_access/hotel/vid19.mp4')
if not v.isOpened():
    print("Error: Unable to open video file.")
    exit()
ret , image = v.read()
if not ret:
    print("Error: Unable to read the first frame from the video.")
    exit()
    
fps = v.get(cv2.CAP_PROP_FPS)

# Get the frame width and height
frame_width = int(v.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(v.get(cv2.CAP_PROP_FRAME_HEIGHT))

In [4]:
cv2.resize(image,(640,640))
cv2.imwrite('C:/Users/HDN/Desktop/Video_survelliance/restricted_access/first_frame.jpg',image)

True

In [5]:
model = YOLO("C:/Users/HDN/Desktop/Video_survelliance/restricted_access/best.pt")

In [6]:
def draw_box(event, x, y, flags, param):
    global drawing, start_x, start_y, end_x, end_y
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        start_x, start_y = x, y
    elif event == cv2.EVENT_LBUTTONUP:
        if drawing:
            end_x, end_y = x, y
            cv2.rectangle(image, (start_x, start_y), (end_x, end_y), (0, 255, 0), 2)
            cv2.imshow("Image", image)
            drawing = False
            print("Bounding Box Coordinates: ", (start_x, start_y), (end_x, end_y))

image_path = 'C:/Users/HDN/Desktop/Video_survelliance/restricted_access/first_frame.jpg'
image = cv2.imread(image_path)
cv2.imshow("Image", image)
cv2.setMouseCallback("Image", draw_box)

drawing = False
start_x, start_y, end_x, end_y = 0, 0, 0, 0

cv2.waitKey(0)
cv2.destroyAllWindows()
r_area = [start_x,start_y,end_x,end_y]

Bounding Box Coordinates:  (4, 5) (388, 333)


In [7]:
def intersect(box1,box2):
    x1_min, y1_min, x1_max, y1_max = box1
    x2_min, y2_min, x2_max, y2_max = box2
    
    intersection_x_min = max(x1_min, x2_min)
    intersection_y_min = max(y1_min, y2_min)
    intersection_x_max = min(x1_max, x2_max)
    intersection_y_max = min(y1_max, y2_max)
    
    intersection_area = max(0, intersection_x_max - intersection_x_min + 1) * max(0, intersection_y_max - intersection_y_min + 1)
    print(intersection_area)
    if intersection_area > 50:
        return True
    else:
        return False

In [8]:
def add_box(img,box):
    x_min, y_min, x_max, y_max = box
    x_min, y_min, x_max, y_max = int(x_min), int(y_min), int(x_max), int(y_max)
    color = (0, 0, 255)  # BGR color format (green)
    thickness = 2  # Thickness of the bounding box
    cv2.rectangle(img, (x_min, y_min), (x_max, y_max), color, thickness)

In [9]:
def test_for_ra(frame):
    global ra
    result = model(frame)
    classes = result[0].boxes.cls
    bounding_boxes = result[0].boxes.xyxy
    for i in range(0,result[0].boxes.cls.size()[0]):
        if(classes[i] == 0):
            if(intersect(bounding_boxes[i],r_area)):
                add_box(frame,bounding_boxes[i])
                ra = True
                print('capture')
#     cv2.imshow("Image", frame)
#     cv2.destroyAllWindows()
    return frame

In [10]:
def frames_to_video():
    global output_path
    global counter
    global output_video
    global last_feed
    output_path = 'C:/Users/HDN/Desktop/Video_survelliance/restricted_access/ra/ra'+str(counter)+'.mp4'
    output_video = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))
    counter +=1
    for frame_ts in last_feed:
        output_video.write(frame_ts[0])
    output_video.release()
    print('vidio_saved')

In [11]:
flag = True
frame_counter = 0
while flag:
        ret, frame = v.read()
        
        if not ret:
            print("No feed")
            break
        
        if(len(last_feed)>=max_frames):
            last_feed = last_feed[1:]
        
        current_time = time.time()
        processed_frame = test_for_ra(frame)
        
        last_feed.append((processed_frame,current_time))        
        
        if(ra and frame_counter%50==0):
            print("!!Alert!!")
            #message_Sending
            frames_to_video()
            ra = False
            
        if cv2.waitKey(1) & 0xFF == ord('q'):
                flag = False
                break
                exit()
        frame_counter += 1
v.release()


0: 384x640 7 persons, 3 waiterss, 93.0ms
Speed: 7.0ms preprocess, 93.0ms inference, 371.2ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(14243.1377, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(3959.4971, device='cuda:0')
capture
tensor(2930.1597, device='cuda:0')
capture
tensor(2513.9604, device='cuda:0')
capture
tensor(0., device='cuda:0')
!!Alert!!
vidio_saved

0: 384x640 7 persons, 4 waiterss, 23.8ms
Speed: 3.9ms preprocess, 23.8ms inference, 4.9ms postprocess per image at shape (1, 3, 384, 640)
tensor(13929.2305, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(3935.8799, device='cuda:0')
capture
tensor(2946.8711, device='cuda:0')
capture
tensor(2404.7544, device='cuda:0')
capture
tensor(0., device='cuda:0')

0: 384x640 7 persons, 3 waiterss, 24.0ms
Speed: 3.9ms preprocess, 24.0ms inference, 5.1ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(14049.6318

0: 384x640 5 persons, 5 waiterss, 24.1ms
Speed: 6.9ms preprocess, 24.1ms inference, 4.5ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(3640.3372, device='cuda:0')
capture
tensor(3396.4438, device='cuda:0')
capture
tensor(2574.1704, device='cuda:0')
capture

0: 384x640 5 persons, 5 waiterss, 25.1ms
Speed: 5.4ms preprocess, 25.1ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(3350.9641, device='cuda:0')
capture
tensor(3610.4607, device='cuda:0')
capture
tensor(2525.5010, device='cuda:0')
capture

0: 384x640 5 persons, 5 waiterss, 24.0ms
Speed: 7.4ms preprocess, 24.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(3348.4980, device='cuda:0')
capture
tensor(3629.5750, device='cuda:0')
capture
tensor(2463.9841, device='cuda:0')
capture

0: 384x640 6 persons, 5 wait

capture
tensor(19.5897, device='cuda:0')
tensor(0., device='cuda:0')

0: 384x640 7 persons, 3 waiterss, 24.0ms
Speed: 5.8ms preprocess, 24.0ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(3578.4546, device='cuda:0')
capture
tensor(2937.6931, device='cuda:0')
capture
tensor(2487.1594, device='cuda:0')
capture
tensor(24.0432, device='cuda:0')
tensor(0., device='cuda:0')

0: 384x640 7 persons, 3 waiterss, 25.0ms
Speed: 4.5ms preprocess, 25.0ms inference, 5.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(3535.9087, device='cuda:0')
capture
tensor(3000.1589, device='cuda:0')
capture
tensor(2500.7515, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(14.7055, device='cuda:0')

0: 384x640 8 persons, 3 waiterss, 24.6ms
Speed: 3.0ms preprocess, 24.6ms inference, 4.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cud

0: 384x640 8 persons, 2 waiterss, 26.0ms
Speed: 5.0ms preprocess, 26.0ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(3864.4434, device='cuda:0')
capture
tensor(3066.9004, device='cuda:0')
capture
tensor(2647.6050, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(2219.9539, device='cuda:0')
capture
tensor(197.6615, device='cuda:0')
capture

0: 384x640 7 persons, 2 waiterss, 27.4ms
Speed: 6.0ms preprocess, 27.4ms inference, 6.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(3039.8770, device='cuda:0')
capture
tensor(3911.9688, device='cuda:0')
capture
tensor(2556.9905, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(2213.0740, device='cuda:0')
capture

0: 384x640 8 persons, 2 waiterss, 27.8ms
Speed: 5.0ms preprocess, 27.8ms inference, 6.7ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
te

capture
tensor(2663.1123, device='cuda:0')
capture
tensor(200.6281, device='cuda:0')
capture

0: 384x640 7 persons, 3 waiterss, 27.0ms
Speed: 3.0ms preprocess, 27.0ms inference, 3.1ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(2293.4517, device='cuda:0')
capture
tensor(2867.9810, device='cuda:0')
capture
tensor(2789.3760, device='cuda:0')
capture
tensor(2713.4751, device='cuda:0')
capture
tensor(166.4044, device='cuda:0')
capture

0: 384x640 7 persons, 3 waiterss, 31.9ms
Speed: 3.0ms preprocess, 31.9ms inference, 4.7ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(2853.8762, device='cuda:0')
capture
tensor(2270.1536, device='cuda:0')
capture
tensor(2809.9343, device='cuda:0')
capture
tensor(2688.3650, device='cuda:0')
capture
tensor(268.3146, device='cuda:0')
capture

0: 384x640 8 persons, 3 waiterss, 28.2ms
Speed: 3.2ms preprocess, 28.2ms inference, 4.0m

...

Speed: 3.2ms preprocess, 25.5ms inference, 4.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(2583.9385, device='cuda:0')
capture
tensor(5091.7773, device='cuda:0')
capture
tensor(16632.3262, device='cuda:0')
capture
tensor(3202.8672, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(9185.2363, device='cuda:0')
capture
tensor(39909.6133, device='cuda:0')
capture

0: 384x640 8 persons, 2 waiterss, 25.4ms
Speed: 4.1ms preprocess, 25.4ms inference, 5.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(2594.2024, device='cuda:0')
capture
tensor(9043.1533, device='cuda:0')
capture
tensor(15585.4629, device='cuda:0')
capture
tensor(3186.1592, device='cuda:0')
capture
tensor(5199.6602, device='cuda:0')
capture
tensor(0., device='cuda:0')

0: 384x640 7 persons, 3 waiterss, 25.0ms
Speed: 6.3ms preprocess, 25.0ms inference, 3.0ms postprocess per image at shap

Speed: 4.0ms preprocess, 27.0ms inference, 4.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(2710.3975, device='cuda:0')
capture
tensor(38303.7070, device='cuda:0')
capture
tensor(2779.8635, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(3258.0088, device='cuda:0')
capture

0: 384x640 7 persons, 1 waiters, 26.0ms
Speed: 5.0ms preprocess, 26.0ms inference, 4.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(2698.6575, device='cuda:0')
capture
tensor(2943.0452, device='cuda:0')
capture
tensor(35446.4336, device='cuda:0')
capture
tensor(3487.4680, device='cuda:0')
capture
tensor(0., device='cuda:0')

0: 384x640 7 persons, 1 waiters, 27.0ms
Speed: 4.0ms preprocess, 27.0ms inference, 4.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(2675.6094, device='cuda:0')
capture
tensor(2884.20

0: 384x640 7 persons, 5 waiterss, 35.0ms
Speed: 4.0ms preprocess, 35.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(2796.3376, device='cuda:0')
capture
tensor(3248.5955, device='cuda:0')
capture
tensor(4055.8059, device='cuda:0')
capture
tensor(19734.9824, device='cuda:0')
capture
tensor(0., device='cuda:0')

0: 384x640 7 persons, 6 waiterss, 34.1ms
Speed: 3.4ms preprocess, 34.1ms inference, 3.9ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(2853.0247, device='cuda:0')
capture
tensor(3103.4978, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(4226.5186, device='cuda:0')
capture
tensor(0., device='cuda:0')

0: 384x640 7 persons, 6 waiterss, 32.0ms
Speed: 4.4ms preprocess, 32.0ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(2924.8669, device='cuda

tensor(3092.3271, device='cuda:0')
capture
tensor(3777.2034, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(9699.5029, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')

0: 384x640 7 persons, 3 waiterss, 29.0ms
Speed: 3.0ms preprocess, 29.0ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(3214.5947, device='cuda:0')
capture
tensor(3844.2585, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(9819.2959, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')

0: 384x640 8 persons, 3 waiterss, 29.2ms
Speed: 6.0ms preprocess, 29.2ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(3252.1055, device='cuda:0')
capture
tensor(3897.9890, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(6752.1704, device='cuda:0')
capture
tensor(0., device='cuda:0')

tensor(2929.4668, device='cuda:0')
capture
tensor(4197.8564, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(8829.2021, device='cuda:0')
capture
tensor(10810.8574, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(1460.5698, device='cuda:0')
capture

0: 384x640 9 persons, 1 waiters, 38.6ms
Speed: 4.0ms preprocess, 38.6ms inference, 4.0ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(4488.3701, device='cuda:0')
capture
tensor(3027.3171, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(9229.1543, device='cuda:0')
capture
tensor(0., device='cuda:0')
tensor(1462.1937, device='cuda:0')
capture

0: 384x640 10 persons, 1 waiters, 40.1ms
Speed: 3.0ms preprocess, 40.1ms inference, 2.9ms postprocess per image at shape (1, 3, 384, 640)
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(4382.0684, device='c